In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 03
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    Prompting Techniques<br>
    <span style="color:#63b3ed;">Zero-Shot to Tree-of-Thought</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    The model is frozen — the only lever you have is the prompt.
    We’ll go from zero-shot to chain-of-thought to self-consistency,
    then compare system prompts and persona effects.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">Prompting</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">Chain-of-Thought</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">Few-Shot</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~30 min</span>
  </div>
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Environment Check</h2>
</div>

First — verify the environment is ready.

In [ ]:
!python3 ../../scripts/setup_check.py


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: Server Discovery &amp; Warmup</h2>
</div>

Discover running MLX servers and warm them up in parallel.

In [ ]:
# Discover running MLX servers and warm up in parallel
from openai import OpenAI
import time
import concurrent.futures
from IPython.display import HTML, display

# Import shared helpers (includes discover_servers)
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../scripts").resolve()))
import notebook_helpers

# Discover servers (reads model ID from process command line, not /v1/models)
print("Discovering MLX servers...", flush=True)
MODELS, clients = notebook_helpers.discover_servers()

if not MODELS:
    raise RuntimeError("No MLX servers found! Start at least one on ports 8800-8802.")

for m in MODELS:
    print(f"  Port {m['port']}: {m['label']} ({m['model']})", flush=True)

# Warm up all discovered models in parallel
def warmup(m):
    t0 = time.time()
    try:
        clients[m["label"]].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=1,
        )
        return m["label"], time.time() - t0, True
    except Exception as e:
        return m["label"], time.time() - t0, False

print(f"\nWarming up {len(MODELS)} models in parallel...", flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    warmup_results = list(ex.map(warmup, MODELS))

# Display status table
rows = ""
for label, elapsed, ok in warmup_results:
    status = "\u2713" if ok else "\u2717"
    color = "#16a34a" if ok else "#dc2626"
    m = next(m for m in MODELS if m["label"] == label)
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{m['port']}</td>"
        f"<td style='padding:6px 12px; color:#374151; font-size:0.85em; border-bottom:1px solid #e5e7eb;'>{m['model']}</td>"
        f"<td style='padding:6px 12px; color:{color}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{status}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{elapsed:.1f}s</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Port</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Model ID</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Status</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Warmup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))
print(f"All {len(MODELS)} models ready!")

notebook_helpers.init(MODELS, clients)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧰 Step 3: Load Helpers</h2>
</div>

Import shared utility functions.

In [ ]:
from notebook_helpers import strip_think, compare_models, show_metrics_table
print("Helpers loaded.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎯 Step 4: Zero-Shot Classification</h2>
</div>

**Zero-shot** = no examples. Just the task description. The model relies entirely on pretraining.

This works well for tasks the model has seen during training, but fails on ambiguous or domain-specific classifications.

In [ ]:
# Zero-shot classification — no examples, just the task
results = compare_models(
    "Classify this support ticket into exactly one category "
    "(billing, technical, account, shipping):\n\n"
    "\"I was charged twice for my subscription last month "
    "and the second charge shows a different amount.\"\n\n"
    "Reply with just the category name.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📚 Step 5: Few-Shot with Examples</h2>
</div>

**Few-shot** = provide labeled examples before the real query. The model learns the pattern from examples without any fine-tuning.

This is like providing test vectors — the model calibrates its output format and decision boundaries from the examples.

In [ ]:
# Few-shot classification — examples teach the pattern
results = compare_models(
    "Classify support tickets into: billing, technical, account, shipping.\n\n"
    "Examples:\n"
    "\"My payment didn't go through\" -> billing\n"
    "\"App crashes on login\" -> technical\n"
    "\"Can't reset my password\" -> account\n"
    "\"Package arrived damaged\" -> shipping\n\n"
    "Now classify:\n"
    "\"I was charged twice for my subscription last month "
    "and the second charge shows a different amount.\"\n\n"
    "Reply with just the category name.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧠 Step 6: Chain-of-Thought Reasoning</h2>
</div>

**Chain-of-Thought (CoT)** = ask the model to show its work. This dramatically improves accuracy on multi-step problems.

The magic phrase: *"Let's think step by step."*

Without CoT, models often jump to answers. With CoT, they decompose the problem — like showing work on a math exam.

In [ ]:
# Chain-of-thought on a multi-step math problem
results = compare_models(
    "A farmer has 3 fields. Field A produces 240 bushels per acre across 5 acres. "
    "Field B produces 180 bushels per acre across 8 acres. "
    "Field C produces 300 bushels per acre across 3 acres. "
    "He sells wheat at $7.50 per bushel but pays $2.25 per bushel in costs. "
    "What is his total profit?\n\n"
    "Let's think step by step.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🔗 Step 7: Few-Shot + CoT Combined</h2>
</div>

The most powerful basic technique: combine few-shot examples with chain-of-thought reasoning traces. Each example shows not just the answer, but *how* to reason.

In [ ]:
# Few-shot CoT — examples include reasoning traces
results = compare_models(
    "Classify the sentiment of product reviews as positive, negative, or mixed.\n"
    "Think through your reasoning before classifying.\n\n"
    "Example 1: \"Battery life is incredible but the screen cracks easily.\"\n"
    "Reasoning: Positive aspect (battery) but negative aspect (screen durability). "
    "Both are significant. -> mixed\n\n"
    "Example 2: \"Absolutely love everything about this product!\"\n"
    "Reasoning: Universally positive language, no caveats. -> positive\n\n"
    "Now classify: \"The firmware update fixed the crashing issue but "
    "introduced a new bug where the UART drops bytes above 115200 baud.\"\n"
    "Show your reasoning, then classify.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎲 Step 8: Self-Consistency (Majority Vote)</h2>
</div>

**Self-consistency** = sample multiple CoT paths and take the majority answer. Like running a test 5 times and going with the mode.

Uses `temperature > 0` to get diverse reasoning paths.

In [ ]:
# Self-consistency — 5 samples with majority vote
import re
from collections import Counter
from IPython.display import HTML, display

prompt = (
    "A store has a 20% off sale. A member gets an additional 10% off the sale price. "
    "The original price is $150. What is the final price? "
    "Think step by step, then give the final answer as: ANSWER: $XX.XX"
)

target = MODELS[0]  # Use first available model
print(f"Self-consistency with {target['label']} (5 samples, temp=0.7)...\n", flush=True)

answers = []
reasoning_snippets = []
for i in range(5):
    response = clients[target["label"]].chat.completions.create(
        model=target["model"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    text = strip_think(response.choices[0].message.content)
    # Extract answer
    match = re.search(r"ANSWER:\s*\$?([\d.]+)", text)
    answer = match.group(1) if match else "???"
    answers.append(answer)
    # Get first line of reasoning
    first_line = text.split("\n")[0][:80]
    reasoning_snippets.append(first_line)
    print(f"  Sample {i+1}: ${answer}  ({first_line}...)", flush=True)

# Majority vote
vote_counts = Counter(answers)
winner, count = vote_counts.most_common(1)[0]

rows = ""
for ans, cnt in vote_counts.most_common():
    is_winner = ans == winner
    bg = "#f0fdf4" if is_winner else "#f9fafb"
    badge = ' <span style="color:#16a34a; font-weight:bold;">[WINNER]</span>' if is_winner else ""
    rows += (
        f"<tr style='background:{bg};'>"
        f"<td style='padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;'>${ans}{badge}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{cnt}/5</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:250px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Answer</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Votes</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">Correct answer: $108.00 (150 \u00d7 0.80 \u00d7 0.90). Self-consistency reduces variance from any single sample.</div>
"""))
print(f"\nMajority vote: ${winner} ({count}/5 agree)")
print(f"Expected: $108.00")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📡 Step 9: System Prompt Effects</h2>
</div>

System prompts set the model's persona and constraints *before* the user message. They’re like configuring a peripheral’s control registers before enabling it.

In [ ]:
# Compare the same question with different system prompts
from IPython.display import HTML, display
import time

question = "What is a mutex?"

system_prompts = {
    "No system prompt": None,
    "ELI5": "You are an expert who explains complex topics to a 5-year-old. Use simple analogies.",
    "Embedded engineer": "You are a senior embedded systems engineer. Be precise, reference hardware, use technical terms.",
    "Haiku poet": "You respond only in haiku (5-7-5 syllable) format. No other text.",
}

target = MODELS[0]
print(f"Comparing system prompts with {target['label']}...\n", flush=True)

rows = ""
for label, sys_prompt in system_prompts.items():
    messages = []
    if sys_prompt:
        messages.append({"role": "system", "content": sys_prompt})
    messages.append({"role": "user", "content": question})
    
    t0 = time.time()
    response = clients[target["label"]].chat.completions.create(
        model=target["model"],
        messages=messages,
        max_tokens=200,
        temperature=0.3,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - t0
    text = strip_think(response.choices[0].message.content).strip()
    # Truncate for display
    display_text = text[:200] + ("..." if len(text) > 200 else "")
    display_text = display_text.replace("\n", "<br>")
    
    rows += (
        f"<tr>"
        f"<td style='padding:8px 12px; color:#2563eb; font-weight:bold; border-bottom:1px solid #e5e7eb; white-space:nowrap;'>{label}</td>"
        f"<td style='padding:8px 12px; color:#374151; border-bottom:1px solid #e5e7eb; font-size:0.85em;'>{display_text}</td>"
        f"<td style='padding:8px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{elapsed:.1f}s</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:sans-serif; width:100%;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">System Prompt</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Response</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))
print("Same question, 4 completely different responses. The system prompt is the most powerful single lever.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎭 Step 10: Persona Showdown</h2>
</div>

Let’s push system prompts further with strong personas and stream all responses in parallel.

In [ ]:
# Persona showdown — 3 wildly different system prompts, same technical question
results_pirate = compare_models(
    "Explain how a watchdog timer prevents embedded systems from hanging.",
    system_prompt=(
        "You are a pirate captain who explains everything using nautical metaphors. "
        "Keep it under 4 sentences."
    ),
    max_tokens=256,
)
show_metrics_table(results_pirate)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">⚠️ Step 11: Common Prompting Pitfalls</h2>
</div>

Reference card for the most common prompting mistakes.

In [ ]:
# Common pitfalls reference card
from IPython.display import HTML, display

display(HTML("""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:sans-serif; width:100%;">
  <thead><tr style="background:#dc2626;">
    <th style="padding:8px 12px; color:white; text-align:left;">Pitfall</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Bad</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Better</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">Vague instructions</td>
        <td style="padding:6px 12px; color:#dc2626; border-bottom:1px solid #e5e7eb;">"Summarize this"</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">"Summarize in 3 bullet points, max 20 words each"</td></tr>
    <tr><td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">No output format</td>
        <td style="padding:6px 12px; color:#dc2626; border-bottom:1px solid #e5e7eb;">"Extract the data"</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">"Extract as JSON: {\"name\": str, \"value\": float}"</td></tr>
    <tr><td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">Negative framing</td>
        <td style="padding:6px 12px; color:#dc2626; border-bottom:1px solid #e5e7eb;">"Don't use jargon"</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">"Use plain language a non-expert can understand"</td></tr>
    <tr><td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">Kitchen sink prompt</td>
        <td style="padding:6px 12px; color:#dc2626; border-bottom:1px solid #e5e7eb;">"Do X and Y and Z and also W"</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">Break into separate calls or numbered steps</td></tr>
    <tr><td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">No temperature control</td>
        <td style="padding:6px 12px; color:#dc2626; border-bottom:1px solid #e5e7eb;">Default temp for classification</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">temp=0 for deterministic, temp=0.7+ for creative</td></tr>
    <tr><td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">Ignoring token budget</td>
        <td style="padding:6px 12px; color:#dc2626; border-bottom:1px solid #e5e7eb;">Huge context, tiny max_tokens</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">Reserve enough tokens for the expected output length</td></tr>
  </tbody>
</table>
"""))

print("Key rule: be specific about format, length, and constraints. The model can't read your mind.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Zero-shot** works for tasks the model has seen in pretraining. Add examples (few-shot) when it doesn’t.
- **Chain-of-Thought** dramatically improves multi-step reasoning — just add "Let's think step by step."
- **Few-shot + CoT** is the sweet spot: examples with reasoning traces calibrate both format and logic.
- **Self-consistency** (majority vote over N samples) reduces variance for critical decisions.
- **System prompts** are the most powerful single lever — they set persona, constraints, and output format.
- **Temperature** controls diversity: 0 for deterministic tasks, 0.7+ for creative or self-consistency sampling.
- Be **specific** about format, length, and constraints. Vague prompts get vague answers.
- Everything here ran locally — no data left your machine.

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What's Next</h2>
</div>

- **Section 04:** Prompt Optimization — DSPy, automated optimization, prompt CI/CD
- **Section 05:** Structured Output — making LLMs return JSON you can actually parse
- **Section 06:** RAG — retrieval-augmented generation with local embeddings